In [1]:
# 1. Cài đặt PostgreSQL
!apt-get update -qq && apt-get install -y -qq postgresql postgresql-contrib

# 2. Khởi động service
!service postgresql start

# 3. Tạo database và cấp quyền (Tạo cả user 'postgres' và 'ticket' để đảm bảo không bị lỗi login DB)
!su - postgres -c "psql -c \"ALTER USER postgres WITH PASSWORD 'tan1042005';\""
!su - postgres -c "psql -c \"CREATE USER ticket WITH PASSWORD '123';\""
!su - postgres -c "psql -c \"CREATE DATABASE ticket OWNER postgres;\""
!su - postgres -c "psql -c \"GRANT ALL PRIVILEGES ON DATABASE ticket TO ticket;\""


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[ OK ] PostgreSQL 16 database server        
ALTER ROLE
ERROR:  role "ticket" already exists
ERROR:  database "ticket" already exists
GRANT


In [2]:
# 1. Cài JDK 21 và Maven
!apt-get install -y -qq openjdk-21-jdk maven

# 2. Xóa thư mục cũ nếu có và clone lại
!rm -rf /kaggle/working/ticket
!git clone https://github.com/Lauyrl/ticket.git /kaggle/working/ticket

# 3. Chuyển vào thư mục dự án
%cd /kaggle/working/ticket


Cloning into '/kaggle/working/ticket'...
remote: Enumerating objects: 581, done.
remote: Counting objects: 100% (581/581), done.
remote: Compressing objects: 100% (302/302), done.
remote: Total 581 (delta 217), reused 498 (delta 148), pack-reused 0 (from 0)
Receiving objects: 100% (581/581), 746.63 KiB | 15.24 MiB/s, done.
Resolving deltas: 100% (217/217), done.
/kaggle/working/ticket


In [3]:
# 1. Đổi Java version từ 25 về 21 trong pom.xml
!sed -i 's/<java.version>25<\/java.version>/<java.version>21<\/java.version>/g' pom.xml

# 2. Đổi host database từ 'postgres' sang 'localhost' trong application.properties
!sed -i 's/postgres:5432/localhost:5432/g' src/main/resources/application.properties

# 3. Tự động sửa SecurityConfig.java (bỏ @RequiredArgsConstructor, thêm constructor tường minh)
security_path = "src/main/java/com/example/ticket/config/SecurityConfig.java"
with open(security_path, "r", encoding="utf-8") as f:
    content = f.read()

# Xóa import Lombok RequiredArgsConstructor
content = content.replace("import lombok.RequiredArgsConstructor;\n", "")

# Thay @RequiredArgsConstructor bằng constructor tường minh
old_class_def = "@RequiredArgsConstructor\npublic class SecurityConfig {"
new_class_def = """public class SecurityConfig {

    public SecurityConfig(JwtAuthenticationFilter jwtAuthenticationFilter) {
        this.jwtAuthenticationFilter = jwtAuthenticationFilter;
    }"""
content = content.replace(old_class_def, new_class_def)

with open(security_path, "w", encoding="utf-8") as f:
    f.write(content)

print(" Đã vá pom.xml, application.properties và SecurityConfig.java thành công!")


 Đã vá pom.xml, application.properties và SecurityConfig.java thành công!


In [4]:
%%writefile src/main/resources/db/migration/V7__seed_load_test_data.sql
-- 1. Insert shared test customer user: testuser@example.com / password123
INSERT INTO users (email, password, full_name, role)
VALUES (
    'testuser@example.com',
    '$2a$10$UmOu.dDFwa3NmiIeRns8v.MAdyMvEuznxwS7mPwUc3GEGKJNAmWxa',
    'Test Customer',
    'CUSTOMER'
)
ON CONFLICT (email) DO NOTHING;

-- 2. Insert 1000 concurrent customer users
INSERT INTO users (email, password, full_name, role)
SELECT
    'user' || i || '@example.com',
    '$2a$10$UmOu.dDFwa3NmiIeRns8v.MAdyMvEuznxwS7mPwUc3GEGKJNAmWxa',
    'Load Test User ' || i,
    'CUSTOMER'
FROM generate_series(1, 1000) AS s(i)
ON CONFLICT (email) DO NOTHING;

-- 3. Create Event 4 for high-capacity load testing
INSERT INTO events (
    name, description, category, venue, city, starts_at, ends_at, image_url, status, max_seats_per_user, hold_duration_minutes, created_by
)
VALUES (
    'Mega Concert 2026 - Load Test Arena',
    'High-capacity stadium concert designed for system load testing and concurrency evaluation.',
    'Music', 'National Stadium Arena', 'Hanoi',
    '2026-12-31 19:00:00+07', '2026-12-31 23:59:00+07', '1', 'ON_SALE', 10, 15, 1
);

DO $$
DECLARE
    v_event_id BIGINT;
    v_zone_vip_id BIGINT;
    v_zone_standard_id BIGINT;
BEGIN
    SELECT id INTO v_event_id FROM events WHERE name = 'Mega Concert 2026 - Load Test Arena' ORDER BY id DESC LIMIT 1;

    IF v_event_id IS NOT NULL THEN
        INSERT INTO seat_zones (event_id, name, price) VALUES (v_event_id, 'VIP Arena', 1000000) ON CONFLICT (event_id, name) DO NOTHING;
        INSERT INTO seat_zones (event_id, name, price) VALUES (v_event_id, 'Standard Arena', 350000) ON CONFLICT (event_id, name) DO NOTHING;

        SELECT id INTO v_zone_vip_id FROM seat_zones WHERE event_id = v_event_id AND name = 'VIP Arena';
        SELECT id INTO v_zone_standard_id FROM seat_zones WHERE event_id = v_event_id AND name = 'Standard Arena';

        -- 200 ghế VIP
        INSERT INTO event_seats (event_id, zone_id, row_index, seat_number)
        SELECT v_event_id, v_zone_vip_id, r, s
        FROM generate_series(1, 10) AS r CROSS JOIN generate_series(1, 20) AS s
        ON CONFLICT (event_id, row_index, seat_number) DO NOTHING;

        -- 800 ghế Standard
        INSERT INTO event_seats (event_id, zone_id, row_index, seat_number)
        SELECT v_event_id, v_zone_standard_id, r, s
        FROM generate_series(11, 30) AS r CROSS JOIN generate_series(1, 40) AS s
        ON CONFLICT (event_id, row_index, seat_number) DO NOTHING;
    END IF;
END $$;

Writing src/main/resources/db/migration/V7__seed_load_test_data.sql


In [5]:
# Biên dịch trực tiếp để thấy rõ quá trình tải thư viện
!mvn clean package -DskipTests

[INFO] Scanning for projects...
[INFO] 
[INFO] -------------------------< com.example:ticket >-------------------------
[INFO] Building  0.0.1-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- maven-clean-plugin:3.5.0:clean (default-clean) @ ticket ---
[INFO] 
[INFO] --- maven-resources-plugin:3.5.0:resources (default-resources) @ ticket ---
[INFO] Copying 1 resource from src/main/resources to target/classes
[INFO] Copying 7 resources from src/main/resources to target/classes
[INFO] 
[INFO] --- maven-compiler-plugin:3.15.0:compile (default-compile) @ ticket ---
[INFO] Recompiling the module because of changed source code.
[INFO] Compiling 69 source files with javac [debug parameters release 21] to target/classes
[INFO] 
[INFO] --- maven-resources-plugin:3.5.0:testResources (default-testResources) @ ticket ---
[INFO] skip non existing resourceDirectory /kaggle/working/ticket/src/test/resources
[INFO] 
[INFO] --- maven-compiler-pl

In [6]:
# Cấp quyền Superuser và toàn quyền trên schema public cho user ticket và postgres
!su - postgres -c "psql -c \"ALTER USER ticket WITH SUPERUSER;\""
!su - postgres -c "psql -c \"ALTER USER postgres WITH SUPERUSER;\""
!su - postgres -c "psql -d ticket -c \"GRANT ALL ON SCHEMA public TO ticket;\""
!su - postgres -c "psql -d ticket -c \"GRANT ALL ON SCHEMA public TO postgres;\""
!su - postgres -c "psql -d ticket -c \"GRANT ALL ON SCHEMA public TO PUBLIC;\""
!su - postgres -c "psql -d ticket -c \"ALTER SCHEMA public OWNER TO ticket;\""


ALTER ROLE
ALTER ROLE
GRANT
GRANT
GRANT
ALTER SCHEMA


In [7]:
import subprocess
import time
import socket

# 1. Chạy file JAR ngầm trên cổng 8081
print("Đang khởi động TicketApplication trên cổng 8081...")
backend_process = subprocess.Popen(
    ["java", "-jar", "target/ticket-0.0.1-SNAPSHOT.jar", "--server.port=8081"],
    stdout=open("backend_runtime.log", "w"),
    stderr=subprocess.STDOUT
)

# 2. Vòng lặp tự động kiểm tra cổng 8081 (tối đa 60 giây)
port_ready = False
for i in range(30):
    time.sleep(2)
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(('127.0.0.1', 8081)) == 0:
            port_ready = True
            break
    print(f"Đang đợi backend mở cổng 8081... ({i*2 + 2}s)")

if port_ready:
    print("✅ SERVER ĐÃ SẴN SÀNG TRÊN CỔNG 8081!")
else:
    print("❌ Quá thời gian chờ hoặc Server gặp lỗi. Dưới đây là log:")
    !tail -n 30 backend_runtime.log

Đang khởi động TicketApplication trên cổng 8081...
Đang đợi backend mở cổng 8081... (2s)
Đang đợi backend mở cổng 8081... (4s)
Đang đợi backend mở cổng 8081... (6s)
Đang đợi backend mở cổng 8081... (8s)
Đang đợi backend mở cổng 8081... (10s)
✅ SERVER ĐÃ SẴN SÀNG TRÊN CỔNG 8081!


In [8]:
!curl -s http://127.0.0.1:8081/api/v1/events | head -c 300

{"content":[{"id":1,"name":"Rock Festival 2026","description":"A live rock music festival featuring local and international bands.","category":"Music","venue":"My Dinh National Stadium","city":"Hanoi","startsAt":"2026-11-15T12:00:00Z","endsAt":"2026-11-15T16:00:00Z","imageUrl":"1","status":"ON_SALE"

In [9]:
!pip install -q locust


In [13]:
%%writefile locustfile.py
import random
from locust import HttpUser, task, between

class ComprehensiveTicketUser(HttpUser):
    # Thời gian nghỉ giữa các hành động từ 0.2s đến 0.8s
    wait_time = between(0.2, 0.8)
    
    # Danh sách Event ID đang mở bán trong DB
    ACTIVE_EVENT_IDS = [1, 3, 4]
    
    def on_start(self):
        """Khởi tạo: Chọn ngẫu nhiên 1 trong 1000 tài khoản mẫu và đăng nhập"""
        self.user_idx = random.randint(1, 1000)
        self.headers = {"Content-Type": "application/json"}
        
        login_res = self.client.post("/api/v1/auth/login", json={
            "email": f"user{self.user_idx}@example.com",
            "password": "password123"
        }, name="POST /api/v1/auth/login")
        
        if login_res.status_code == 200:
            token = login_res.json().get("accessToken")
            self.headers["Authorization"] = f"Bearer {token}"

    # ==========================================================
    # 1. NHÓM TÌM KIẾM & XEM SỰ KIỆN (READ - Trọng số cao nhất)
    # ==========================================================

    @task(6)
    def browse_events(self):
        """Duyệt danh sách sự kiện có phân trang"""
        page = random.choice([0, 1])
        self.client.get(f"/api/v1/events?page={page}&size=10", name="GET /api/v1/events")

    @task(3)
    def search_events_by_city(self):
        """Tìm kiếm sự kiện theo thành phố"""
        self.client.get("/api/v1/events?city=Hanoi&page=0&size=5", name="GET /api/v1/events?city=Hanoi")

    @task(4)
    def view_event_detail(self):
        """Xem trang chi tiết một sự kiện ngẫu nhiên"""
        event_id = random.choice(self.ACTIVE_EVENT_IDS)
        self.client.get(f"/api/v1/events/{event_id}", name="GET /api/v1/events/{eventId}")

    @task(5)
    def view_seats_layout(self):
        """Xem sơ đồ ghế và tình trạng ghế trống"""
        event_id = random.choice(self.ACTIVE_EVENT_IDS)
        self.client.get(f"/api/v1/events/{event_id}/seats", headers=self.headers, name="GET /api/v1/events/{eventId}/seats")

    # ==========================================================
    # 2. NHÓM TRANG CÁ NHÂN & LỊCH SỬ (DASHBOARD)
    # ==========================================================

    @task(2)
    def get_my_profile(self):
        """Xem thông tin tài khoản hiện tại"""
        self.client.get("/api/v1/auth/me", headers=self.headers, name="GET /api/v1/auth/me")

    @task(2)
    def view_my_orders(self):
        """Xem lịch sử các đơn hàng của tôi"""
        self.client.get("/api/v1/orders/my?page=0&size=10", headers=self.headers, name="GET /api/v1/orders/my")

    @task(2)
    def view_my_active_holds(self):
        """Kiểm tra các ghế mình đang tạm giữ trong Event 4"""
        self.client.get("/api/v1/events/4/holds", headers=self.headers, name="GET /api/v1/events/{eventId}/holds")

    # ==========================================================
    # 3. NHÓM THAO TÁC GIỮ GHẾ RỒI HỦY (HOLD & RELEASE)
    # ==========================================================

    @task(2)
    def hold_and_cancel_seat(self):
        """Giữ ghế nhưng sau đó đổi ý bấm Hủy giữ chỗ"""
        seat_id = random.randint(196, 1195)
        hold_url = f"/api/v1/events/4/seats/{seat_id}/hold"
        
        # 1. Bấm giữ ghế
        with self.client.post(hold_url, headers=self.headers, catch_response=True, name="POST /seats/{id}/hold") as hold_res:
            # 200 (OK) hoặc 400/409 (Bị tranh ghế) đều tính là hệ thống xử lý đúng, không bị crash
            if hold_res.status_code in [200, 400, 409]: 
                hold_res.success()
                if hold_res.status_code == 200:
                    # 2. Chỉ hủy nếu giữ thành công
                    self.client.delete(hold_url, headers=self.headers, name="DELETE /seats/{id}/hold")
            else:
                # Chỉ báo lỗi Load Test nếu Server bị 500, timeout, v.v.
                hold_res.failure(f"Lỗi Server: {hold_res.status_code}")

    # ==========================================================
    # 4. QUY TRÌNH MUA VÉ TRỌN GÓI (FULL PURCHASE FLOW)
    # ==========================================================

    @task(3)
    def full_booking_flow(self):
        """Quy trình đầy đủ: Giữ ghế -> Tạo đơn hàng -> Thanh toán"""
        seat_id = random.randint(196, 1195)
        hold_url = f"/api/v1/events/4/seats/{seat_id}/hold"
        
        # Bước 1: Giữ ghế
        with self.client.post(hold_url, headers=self.headers, catch_response=True, name="POST /seats/{id}/hold") as hold_res:
            if hold_res.status_code in [400, 409]:
                hold_res.success() # Đánh dấu hệ thống xử lý tranh chấp tốt
                return
            elif hold_res.status_code != 200:
                hold_res.failure(f"Lỗi giữ ghế: {hold_res.status_code}")
                return
            
            hold_res.success()
            hold_data = hold_res.json()
            hold_id = hold_data.get("id")

        if not hold_id:
            return

        # Bước 2: Tạo đơn hàng từ hold_id
        order_payload = {"holdId": hold_id}
        with self.client.post("/api/v1/orders", json=order_payload, headers=self.headers, catch_response=True, name="POST /api/v1/orders") as order_res:
            if order_res.status_code != 201:
                order_res.failure(f"Lỗi tạo đơn: {order_res.status_code}")
                return
            
            order_res.success()
            order_id = order_res.json().get("id")

        if not order_id:
            return

        # Bước 3: Thanh toán đơn hàng
        pay_url = f"/api/v1/orders/{order_id}/pay"
        with self.client.post(pay_url, headers=self.headers, catch_response=True, name="POST /api/v1/orders/{id}/pay") as pay_res:
            if pay_res.status_code == 200:
                pay_res.success()
            else:
                pay_res.failure(f"Lỗi thanh toán: {pay_res.status_code}")

Overwriting locustfile.py


In [14]:
# Kịch bản 1: 100 User (Spawn 10 user/giây)
print("--- ĐANG CHẠY KỊCH BẢN 100 USER ---")
!locust -f locustfile.py --headless --host="http://127.0.0.1:8081" -u 100 -r 10 --run-time 1m --only-summary --csv=results_100

# Kịch bản 2: 500 User (Spawn 50 user/giây)
print("--- ĐANG CHẠY KỊCH BẢN 500 USER ---")
!locust -f locustfile.py --headless --host="http://127.0.0.1:8081" -u 500 -r 50 --run-time 1m --only-summary --csv=results_500

# Kịch bản 3: 1000 User (Spawn 100 user/giây)
print("--- ĐANG CHẠY KỊCH BẢN 1000 USER ---")
!locust -f locustfile.py --headless --host="http://127.0.0.1:8081" -u 1000 -r 100 --run-time 1m --only-summary --csv=results_1000

--- ĐANG CHẠY KỊCH BẢN 100 USER ---
[2026-10-29 16:12:31,853] d295fcb25934/INFO/locust.main: Starting Locust 2.47.0
[2026-10-29 16:12:31,854] d295fcb25934/INFO/locust.main: Run time limit set to 60 seconds
[2026-10-29 16:12:31,855] d295fcb25934/INFO/locust.runners: Ramping to 100 users at a rate of 10.00 per second
[2026-10-29 16:12:40,872] d295fcb25934/INFO/locust.runners: All users spawned: {"ComprehensiveTicketUser": 100} (100 total users)
[2026-10-29 16:13:31,594] d295fcb25934/INFO/locust.main: --run-time limit reached, shutting down
[2026-10-29 16:13:31,638] d295fcb25934/INFO/locust.main: Shutting down (exit code 0)
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
DELETE   DELETE /seats/{id}/hold     255     0(0.00%) |      6       3      38      5 |    4.27        0.00
GET      GET /api/v1/auth/me     753     0(0.00%) |     12       1     428      3 |   12.

In [15]:
import pandas as pd
from IPython.display import display

columns_to_show = ['Type', 'Name', 'Request Count', 'Failure Count', 'Median Response Time', 'Average Response Time', 'Requests/s']

try:
    print("--- KẾT QUẢ 100 USERS ---")
    df_100 = pd.read_csv("results_100_stats.csv")
    display(df_100[columns_to_show])
except FileNotFoundError:
    print("Chưa có file kết quả 100 users.")

try:
    print("--- KẾT QUẢ 500 USERS ---")
    df_500 = pd.read_csv("results_500_stats.csv")
    display(df_500[columns_to_show])
except FileNotFoundError:
    print("Chưa có file kết quả 500 users.")

try:
    print("--- KẾT QUẢ 1000 USERS ---")
    df_1000 = pd.read_csv("results_1000_stats.csv")
    display(df_1000[columns_to_show])
except FileNotFoundError:
    print("Chưa có file kết quả 1000 users.")

--- KẾT QUẢ 100 USERS ---


,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Requests/s
0,DELETE,DELETE /seats/{id}/hold,252,0,5,6.242650,4.262369
1,GET,GET /api/v1/auth/me,739,0,3,13.112689,12.499567
2,GET,GET /api/v1/events,2155,0,3,12.595734,36.450024
3,GET,GET /api/v1/events/{eventId},1435,0,3,13.371360,24.271825
4,GET,GET /api/v1/events/{eventId}/holds,738,0,3,11.101473,12.482653
5,GET,GET /api/v1/events/{eventId}/seats,1837,0,7,18.070833,31.071319
6,GET,GET /api/v1/events?city=Hanoi,1128,0,3,13.059445,19.079177
7,GET,GET /api/v1/orders/my,741,0,4,12.374054,12.533396
8,POST,POST /api/v1/auth/login,100,0,460,601.934648,1.691416
9,POST,POST /api/v1/orders,389,0,8,9.946733,6.579610


--- KẾT QUẢ 500 USERS ---


,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Requests/s
0,DELETE,DELETE /seats/{id}/hold,351,0,240,494.985991,5.907748
1,GET,GET /api/v1/auth/me,2019,0,200,318.929806,33.982173
2,GET,GET /api/v1/events,6066,0,200,309.858947,102.098000
3,GET,GET /api/v1/events/{eventId},3969,0,190,304.677609,66.802994
4,GET,GET /api/v1/events/{eventId}/holds,2024,0,200,332.122586,34.066329
5,GET,GET /api/v1/events/{eventId}/seats,4939,0,220,339.379240,83.129248
6,GET,GET /api/v1/events?city=Hanoi,2891,0,190,288.262478,48.658971
7,GET,GET /api/v1/orders/my,2064,0,210,330.063352,34.739577
8,POST,POST /api/v1/auth/login,500,0,2400,3398.381810,8.415595
9,POST,POST /api/v1/orders,550,0,230,411.424492,9.257155


--- KẾT QUẢ 1000 USERS ---


,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Requests/s
0,DELETE,DELETE /seats/{id}/hold,38,0,910,2560.764660,0.611436
1,GET,GET /api/v1/auth/me,1535,0,710,1170.651869,24.698798
2,GET,GET /api/v1/events,4662,0,710,1361.688535,75.013547
3,GET,GET /api/v1/events/{eventId},3125,0,720,1373.323105,50.282569
4,GET,GET /api/v1/events/{eventId}/holds,1620,0,720,1358.950728,26.066484
5,GET,GET /api/v1/events/{eventId}/seats,3925,0,830,1435.858388,63.154906
6,GET,GET /api/v1/events?city=Hanoi,2421,0,710,1329.959179,38.954912
7,GET,GET /api/v1/orders/my,1540,0,720,1281.895120,24.779250
8,POST,POST /api/v1/auth/login,1000,0,11000,10341.815637,16.090422
9,POST,POST /api/v1/orders,46,0,880,950.824627,0.740159


In [16]:
import os
import zipfile
from IPython.display import FileLink, display

# Danh sách các file kết quả có thể được tạo ra
files_to_download = [
    "results_100_stats.csv", 
    "results_500_stats.csv", 
    "results_1000_stats.csv"
]
zip_filename = "Locust_LoadTest_Results.zip"

# Nén các file hiện có vào 1 file ZIP
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_download:
        if os.path.exists(file):
            zipf.write(file)
            print(f"Đã nén: {file}")
        else:
            print(f"Bỏ qua (không tồn tại): {file}")

# Tạo link tải ngay bên dưới Cell
if os.path.exists(zip_filename):
    print("\n👇 Click vào link bên dưới để tải toàn bộ kết quả về máy:")
    display(FileLink(zip_filename))
else:
    print("\n❌ Không tìm thấy file kết quả nào để tải về.")

Đã nén: results_100_stats.csv
Đã nén: results_500_stats.csv
Đã nén: results_1000_stats.csv

👇 Click vào link bên dưới để tải toàn bộ kết quả về máy:


/kaggle/working/ticket/Locust_LoadTest_Results.zip